# 가설 검정

## 독립 표본 검정

- 두 그룹(표본)간의 평균이 서로 다름을 판단하는 통계 방법
- A그룹과 B그룹을 나누고, 각 그룹을 대상으로 적절한 통계적 검정을 통해 두 그룹 사이에 통계적으로 유의미한 차이가 있는지 없는지 결론을 내리는 것
- "통계적으로 유의미하다"라는 표현을 사용하는 이유 : 우연히 발생할 수 있는 차이와 실제로 의미 있는 차이를 구분하기 위해서이다

### ttest_ind(첫번째 모집단에서 뽑은 표본 데이터, 두번째 모집단에서 뽑은 표본 데이터, 대립가설 정의, 두 모집단의 분산-같다(기본값))

- 양측 검정
    - 귀무가설 : 반별 수학 평균 점수는 같다
    - 대립가설 : 반별 수학 평균 점수는 다르다

In [1]:
import pandas as pd
# 서로 다른 두 반의 점수 (표본 크기가 달라도 상관없음: 8명 vs 5명)
class1 = [85, 90, 92, 88, 86, 89, 83, 87]
class2 = [80, 82, 88, 85, 84]

In [2]:
from scipy.stats import ttest_ind

ttest_ind(class1, class2)

TtestResult(statistic=2.2108140580092237, pvalue=0.04914857789252186, df=11.0)

In [3]:
# 모분산이 다르다면 equal_val=False
ttest_ind(class1, class2, equal_var=False)

TtestResult(statistic=2.1818699281825236, pvalue=0.059589330071355334, df=8.272682358753572)

- 단측 검정 (전제조건. 모분산은 동일하다)
    - 귀무가설 : 반별 수학 평균 점수는 같다
    - 대립가설 : 2반 수학 평균 점수가 더 높다

In [4]:
ttest_ind(class1, class2, alternative='less', equal_var=True)

TtestResult(statistic=2.2108140580092237, pvalue=0.9754257110537391, df=11.0)

- 단측 검정 (전제조건. 모분산은 동일하다)
    - 귀무가설 : 반별 수학 평균 점수는 같다
    - 대립가설 : 1반 수학 평균 점수가 더 높다

In [6]:
ttest_ind(class1, class2, alternative='greater', equal_var=True)

TtestResult(statistic=2.2108140580092237, pvalue=0.02457428894626093, df=11.0)

### 비모수 검정
- 정규성에 위배
- 샤피로-윌크 검정을 먼저 수행 → 정규성 가정이 위배되면 → 맨-휘트니 U 검정을 수행

In [13]:
import pandas as pd
class1 = [85, 90, 92, 88, 86, 89, 83, 87]
class2 = [80, 82, 88, 85, 130]

In [14]:
from scipy import stats

stats.shapiro(class1)

ShapiroResult(statistic=0.9981893537736595, pvalue=0.999986994137081)

In [15]:
stats.shapiro(class2)

ShapiroResult(statistic=0.6880497349322277, pvalue=0.007151570728885509)

In [16]:
# class2가 정규성을 만족하지 않았다 → p < 0.05 → t-검정 대신 Mann-Whitney U 검정(비모수, 독립표본)
# "class1이 class2보다 작다" 검정
stats.mannwhitneyu(class1, class2, alternative='less')

MannwhitneyuResult(statistic=26.0, pvalue=0.8299904236851448)

- 독립표본에서 정규성이 깨지면 → Mann-Whitney U 검정
- 대응표본에서 정규성이 깨지면 → Wilcoxon 검정

In [17]:
result = stats.mannwhitneyu(class1, class2, alternative='less').pvalue

# format(값, '.소수자리수f)
format(result, '.2f')

'0.83'

# 분산 분석(ANOVA)

- 분산 분석(ANOVA)은 여러 집단의 평균 차이가 통계적으로 유의미한지 검정하는 방법
- 주로 3개 이상의 집단을 비교할 때 사용
- (2개 그룹비교라면 t-검정 사용)

- 일원 분산 분석(One-way ANOVA)
    - 단일 요인(독립변수 1개)의 수준 간 평균 차이를 검정
- 이원 분산 분석(Two-way ANOVA)
    - 두 요인(독립변수 2개)의 수준 간, 그들의 상호작용이 평균에 미치는 영향을 검정

- 아래 단어들은 모두 같다
    - 통계학 → 독립변수, 요인(factor)
    - 머신러닝 → 피처(feature), 변수 (입력에 해당되는 컬럼)

- 왜 t-검정을 여러번 반복하지 않고 ANOVA를 사용하는가?
    - 독립표본 t-검정을 집단 쌍마다 여러번 반복하면 1종 오류(귀무가설이 참인데 잘못해서 기각으로 판단하는 오류)가 누적되어 커지는 문제 발생
    - 연구 질문이 하나라면 이를 한번의 통계적 검정으로 실시하는 것이 오류율을 통제하는 데 유리
    - ANOVA는 각 요인의 주 효괴(main effect)와 상호작용 효과(interaction effect)를 함께 검정할 수 있는 장점이 있다

## 일원 분산 분석(One-way ANOVA)

- 집단을 나누는 요인이 1개이고, 그 집단의 수가 3개 이상일 때 사용

- 기본 가정
    - 독립성 : 각 집단의 관측치는 다른 집단의 관측치와 독립적이다 (설계 단계에서 확보하는 기본 가정)
    - 정규성 : 각 집단의 관측치는 정규분포를 따른다 (stats.shapiro())
    - 등분산성 : 모든 집단의 관측치는 동일한 분산을 가진다 (stats.levene())

- 귀무가설과 대립가설 (일원 분산 분석 공통 형태)
    - 귀무가설 : 모든 집단의 평균은 동일하다
    - 대립가설 : 집단의 평균에는 차이가 있다 (모두 다르다는 뜻이 아니라, 적어도 두 그룹 간에는 차이가 있다)

In [18]:
import pandas as pd

# A, B, C, D 4개 그룹(예: 4가지 비료/조건)의 측정값을 담은 데이터프레임 생성
# 분산분석의 목적: "A/B/C/D 네 그룹의 평균이 서로 통계적으로 다른가?"를 검정
df = pd.DataFrame({
    'A': [10.5, 11.3, 10.8, 9.6, 11.1, 10.2, 10.9, 11.4, 10.5, 10.3],
    'B': [11.9, 12.4, 12.1, 13.2, 12.5, 11.8, 12.2, 12.9, 12.4, 12.3],
    'C': [11.2, 11.7, 11.6, 10.9, 11.3, 11.1, 10.8, 11.5, 11.4, 11.0],
    'D': [9.8, 9.4, 9.1, 9.5, 9.6, 9.9, 9.2, 9.7, 9.3, 9.4]
})

# 상위 2개 행만 확인 (데이터 구조 파악용)
print(df.head(2))

      A     B     C    D
0  10.5  11.9  11.2  9.8
1  11.3  12.4  11.7  9.4


### 사전 가정 검정 - 분산 분석 순서

In [20]:
from scipy import stats

print('=== 정규성 검정 ===')
# shapiro() : 각 그룹이 정규분포를 따르는지 검정 (귀무가설 : 정규분포를 따른다)
# p-value가 0.05보다 크면 "정규분포를 따른다"라고 판단 (귀무가설 채택)
print(stats.shapiro(df['A']))
print(stats.shapiro(df['B']))
print(stats.shapiro(df['C']))
print(stats.shapiro(df['D']))

=== 정규성 검정 ===
ShapiroResult(statistic=0.9649054066073813, pvalue=0.8400161543468654)
ShapiroResult(statistic=0.9468040874196029, pvalue=0.6308700692815115)
ShapiroResult(statistic=0.9701646110856055, pvalue=0.892367306190296)
ShapiroResult(statistic=0.9752339025839644, pvalue=0.9346854448707653)


In [22]:
print('=== 등분산 검정 ===')
# levene() : 여러 그룹의 분산이 동일한지 검정 (귀무가설 : 분산이 모두 같다)
# ANOVA는 "그룹 간 분산이 같다"는 전제 필요
# p-value가 0.05보다 크면 "분산이 같다"라고 판단 (귀무가설 채택)
print(stats.levene(df['A'], df['B'], df['C'], df['D']))

=== 등분산 검정 ===
LeveneResult(statistic=1.9355354288758708, pvalue=0.14127835331346628)


In [23]:
print('=== 일원 분산 분석 ===')
# f_oneway() : 실제 일원 분산 분석 수해
# 귀무가설(H0) : 네 그룹(A, B, C, D)의 평균이 모두 같다
# 대립가설(H1) : 적어도 한 그룹의 평균은 다르다
# 결과 : F-통계량, p-value 반환 → p-value < 0.05 → 귀무가설 기각
stats.f_oneway(df['A'], df['B'], df['C'], df['D'])

=== 일원 분산 분석 ===


F_onewayResult(statistic=89.12613851177174, pvalue=1.0018381522523723e-16)